In [5]:
#Basic NN Foundational excercise

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [6]:
# Sample data 
documents = [
    "red apple",
    "green apple",
    "red mango",
    "yellow mango",
    "small apple",
    "large apple",
    "small mango",
    "large mango"
]

prices = [50, 60, 80, 90, 40, 70, 70, 100]

In [7]:
# Convert text into numbers
vectorizer = CountVectorizer()  # A neural network cannot directly understand: red apple so vectorize it . E.G. apple = 1, Green = 0 "red apple" = [1, 0, 0, 0, 1, 0, 0]
X = vectorizer.fit_transform(documents) #Take the text in documents, learn the words, and convert each piece of text into numbers.

In [9]:
print("Words learned by vectorizer:")
print(vectorizer.get_feature_names_out())

print("\nText converted into numbers:")
print(X.toarray())

Words learned by vectorizer:
['apple' 'green' 'large' 'mango' 'red' 'small' 'yellow']

Text converted into numbers:
[[1 0 0 0 1 0 0]
 [1 1 0 0 0 0 0]
 [0 0 0 1 1 0 0]
 [0 0 0 1 0 0 1]
 [1 0 0 0 0 1 0]
 [1 0 1 0 0 0 0]
 [0 0 0 1 0 1 0]
 [0 0 1 1 0 0 0]]


In [10]:
# PyTorch as the engine that runs and trains the neural network. 
# Numpy style - [1, 0, 0, 1] and Tensor style is   [1., 0., 0., 1.] -- Floating point X_tensor . The neural network can now work with X_tensor.
# Similarly for y-tensor - [[50.],[60.],[80.]]
# X_tensor = QUESTIONS / INPUTS and y_tensor = CORRECT ANSWERS

X_tensor = torch.FloatTensor(X.toarray())
y_tensor = torch.FloatTensor(prices).unsqueeze(1)

In [11]:
# STEP 4: Split into training and validation data 

X_train, X_val, y_train, y_val = train_test_split(
    X_tensor,
    y_tensor,
    test_size=0.25,
    random_state=42
) 

In [12]:
# STEP 5: Create batches 

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(  #DataLoader is simply a helper that gives your training data to the neural network in small batches.
    train_dataset,
    batch_size=2,
    shuffle=True
)

In [13]:
# STEP 6: Create Neural Network Structure 

class MySimpleNN(nn.Module):         # create a NN
    def __init__(self, input_size):  # When I create this neural network, tell me how many numbers will come into it. E.g Create a neural network that receives 7 numbers

        super().__init__()                      # Initialize the neural-network machinery
        self.layer1 = nn.Linear(input_size, 8)  # Create a layer that takes the input numbers and produces 8 numbers produced by 8 neurons.
        self.layer2 = nn.Linear(8, 4)           # Take the 8 numbers from Layer 1 and produce 4 new numbers.
        self.layer3 = nn.Linear(4, 1)           # Take the 4 numbers and produce 1 final number (predicted price).
        self.relu = nn.ReLU()                   # A simple mathematical filter that helps the neural network learn more complicated patterns
                                                # nn.Linear(A, B) - Take A numbers → produce B numbers.
    def forward(self, x):                       # When we give some input x to Pytorch, this is how it process it.

        x = self.relu(self.layer1(x))           # Take x → send it through Layer 1 → apply ReLU → call the result x again.
        x = self.relu(self.layer2(x))
        x = self.layer3(x)                      # No Relu here because Layer 3 is the final output layer, and in our example we want it to produce a price
        return x


# Number of input features
input_size = X_train.shape[1]                   # "red apple" = [1, 0, 0, 0, 1, 0, 0]  so input_size = 7. "How many columns/features does my input data have?"= 7
model = MySimpleNN(input_size)                  
print("\nInput size:", input_size)


Input size: 7


In [14]:
# STEP 7: Loss function and optimizer 

loss_function = nn.MSELoss()                    # MSELoss is like the teacher measuring how wrong the student was. E.g Actual price = ₹100 and Model prediction = ₹80 then the model was wrong by ₹20 (Error score)
optimizer = optim.Adam(                         # Use Adam to adjust the model's weights so that the error becomes smaller.
    model.parameters(),                         # Give Adam all the weights and biases inside my neural network.
    lr=0.01                                     # Learning rate size. E.g tiny step → tiny step → tiny step
)

In [ ]:
# STEP 8: Train the neural network 
EPOCHS = 100                                     # Let the model study the training data 100 times.
for epoch in range(EPOCHS):

    model.train()                                # Put the model into training mode.
    for batch_X, batch_y in train_loader:        # batch_X = input data and batch_y = correct answers/prices
        optimizer.zero_grad()                    # Forget the previous batch's adjustment information.
        outputs = model(batch_X)                 # Make a prediction
        loss = loss_function(outputs, batch_y)   # Measure error
        loss.backward()                          # Look backward through the neural network and figure out which weights contributed to the mistake and how they should change
        optimizer.step()                         # Change the weights slightly - PyTorch/Adam automatically learns the weights during training.

    model.eval()                                 # After the model has finished learning from the training data: lets test the model 
    with torch.no_grad():                        # We're only testing, so don't calculate learning/gradient information.
        val_outputs = model(X_val)               # The model predicts prices for the data it didn't train on.
        val_loss = loss_function(                # calculate the validation error.
            val_outputs,
            y_val
        )

    if (epoch + 1) % 10 == 0:                    # Only print the result every 10 rounds.
        print(                                   # Is my model getting better?
            f"Epoch {epoch+1:3d} "                          # E.g. Epoch 10   Training Loss: 250.32   Validation Loss: 300.45
            f"Training Loss: {loss.item():8.2f} "           #      Epoch 20   Training Loss: 180.21   Validation Loss: 220.31
            f"Validation Loss: {val_loss.item():8.2f}"      #      Epoch 30   Training Loss: 120.15   Validation Loss: 160.22
        )

# STEP 9: Create a prediction function
def predict_price(description):
    model.eval()

    with torch.no_grad():                                   # We're not training, so don't waste effort calculating learning information.
        vector = vectorizer.transform([description])
        vector = torch.FloatTensor(vector.toarray())
        prediction = model(vector)
        price = prediction[0].item()
    return max(0, price)                                    # Don't allow a negative price

# STEP 10: Try new products
print("\nPredictions:")
print(
    "red apple →",
    predict_price("red apple")
)
print(
    "large mango →",
    predict_price("large mango")
)
print(
    "small apple →",
    predict_price("small apple")
)

Epoch  10 Training Loss:   583.49 Validation Loss:   378.81
Epoch  20 Training Loss:   663.87 Validation Loss:   254.84
Epoch  30 Training Loss:  1042.31 Validation Loss:   164.00
Epoch  40 Training Loss:   668.75 Validation Loss:   101.59
Epoch  50 Training Loss:   884.53 Validation Loss:    61.30
Epoch  60 Training Loss:   101.60 Validation Loss:    38.39
Epoch  70 Training Loss:   111.08 Validation Loss:    27.59
Epoch  80 Training Loss:   426.06 Validation Loss:    24.99
Epoch  90 Training Loss:   656.15 Validation Loss:    27.12
Epoch 100 Training Loss:   253.83 Validation Loss:    31.91
Epoch 110 Training Loss:   637.40 Validation Loss:    37.58
Epoch 120 Training Loss:   243.05 Validation Loss:    43.23
Epoch 130 Training Loss:   648.72 Validation Loss:    48.84
Epoch 140 Training Loss:   637.51 Validation Loss:    52.72
Epoch 150 Training Loss:   431.27 Validation Loss:    56.87
Epoch 160 Training Loss:   465.42 Validation Loss:    59.69
Epoch 170 Training Loss:   523.29 Valida